In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("HousingData.xls")
df.head()
df.shape
print(df.isnull().sum())

CRIM       20
ZN         20
INDUS      20
CHAS       20
NOX         0
RM          0
AGE        20
DIS         0
RAD         0
TAX         0
PTRATIO     0
B           0
LSTAT      20
MEDV        0
dtype: int64


In [9]:
cols_with_null = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'AGE', 'LSTAT']
for col in cols_with_null:
    df[col] = df[col].fillna(df[col].mean())

print("Missing values after fill:")
print(df.isnull().sum())

Missing values after fill:
CRIM       0
ZN         0
INDUS      0
CHAS       0
NOX        0
RM         0
AGE        0
DIS        0
RAD        0
TAX        0
PTRATIO    0
B          0
LSTAT      0
MEDV       0
dtype: int64


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Single feature (RM = average rooms per dwelling)
X_simple = df[['RM']]
y = df['MEDV']

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)

simple_model = LinearRegression()
simple_model.fit(X_train_s, y_train_s)

y_pred_s = simple_model.predict(X_test_s)

r2_s = r2_score(y_test_s, y_pred_s)
rmse_s = np.sqrt(mean_squared_error(y_test_s, y_pred_s))

print("=" * 50)
print("SIMPLE LINEAR REGRESSION (RM -> MEDV)")
print("=" * 50)
print(f"R² Score: {r2_s:.4f}")
print(f"RMSE: {rmse_s:.2f} ($1,000s)")
print(f"Slope (Coefficient): {simple_model.coef_[0]:.4f}")
print(f"Intercept: {simple_model.intercept_:.4f}")

SIMPLE LINEAR REGRESSION (RM -> MEDV)
R² Score: 0.1610
RMSE: 5.80 ($1,000s)
Slope (Coefficient): 6.6543
Intercept: -20.3424


In [18]:
X_multi = df.drop('MEDV', axis=1)
y = df['MEDV']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y, test_size=0.2, random_state=42
)

multi_model = LinearRegression()
multi_model.fit(X_train_m, y_train_m)

y_pred_m = multi_model.predict(X_test_m)

r2_m = r2_score(y_test_m, y_pred_m)
rmse_m = np.sqrt(mean_squared_error(y_test_m, y_pred_m))

print("=" * 50)
print("MULTIPLE LINEAR REGRESSION - RESULTS")
print(f"R² Score: {r2_m:.4f}")
print(f"RMSE: {rmse_m:.2f} ($1,000s)")

importance = pd.DataFrame({
    'Feature': X_multi.columns,
    'Coefficient': multi_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nTop 5 most important features:")
print(importance.head(5))

MULTIPLE LINEAR REGRESSION - RESULTS
R² Score: 0.7836
RMSE: 2.95 ($1,000s)

Top 5 most important features:
           Feature  Coefficient
4              NOX    -8.046361
13  PRICE_CATEGORY     4.550031
5               RM     1.926656
7              DIS    -0.608310
10         PTRATIO    -0.481014


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Convert target to binary classification
median_price = df['MEDV'].median()
df['PRICE_CATEGORY'] = (df['MEDV'] > median_price).astype(int)

X_cls = df.drop(['MEDV', 'PRICE_CATEGORY'], axis=1)
y_cls = df['PRICE_CATEGORY']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

# Feature scaling for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_c)
X_test_scaled = scaler.transform(X_test_c)

log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train_c)

y_pred_c = log_model.predict(X_test_scaled)

print("=" * 50)
print("LOGISTIC REGRESSION - RESULTS")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_c):.4f} ({accuracy_score(y_test_c, y_pred_c)*100:.2f}%)")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test_c, y_pred_c)}")
print(f"\nClassification Report:\n{classification_report(y_test_c, y_pred_c)}")

LOGISTIC REGRESSION - RESULTS
Accuracy: 0.8191 (81.91%)

Confusion Matrix:
[[37 11]
 [ 6 40]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.77      0.81        48
           1       0.78      0.87      0.82        46

    accuracy                           0.82        94
   macro avg       0.82      0.82      0.82        94
weighted avg       0.82      0.82      0.82        94



In [21]:
q1 = df['MEDV'].quantile(0.25)
q3 = df['MEDV'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f"Before removing outliers: {len(df)} rows")
df = df[(df['MEDV'] >= lower) & (df['MEDV'] <= upper)]
print(f"After removing outliers: {len(df)} rows")

Before removing outliers: 460 rows
After removing outliers: 457 rows
